In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/google/gemma-3-4b-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/gemma-3-4b-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="google/gemma-3-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("google/gemma-3-4b-it")
model = AutoModelForImageTextToText.from_pretrained("google/gemma-3-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/google/gemma-3-4b-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/gemma-3-4b-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login(new_session=False)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="google/gemma-3-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

In [ ]:
!pip install -q huggingface_hub -U



In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
from huggingface_hub import InferenceClient

HF_TOKEN = os.environ.get("HF_TOKEN")  #used login(), this will be set
client = InferenceClient(model="moonshotai/Kimi-K2.5", token=HF_TOKEN)

In [ ]:
SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Example:
{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}

Always use the format:
Question: ...
Thought: ...
Action:
$JSON_BLOB
Observation: ...
Thought: I now know the final answer
Final Answer: ...
"""

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

In [ ]:
output = client.chat.completions.create(
    messages=messages,
    max_tokens=150,
    stop=["Observation:"],
    extra_body={'thinking': {'type': 'disabled'}},
)

print("Model output before Observation:\n", output.choices[0].message.content)

In [ ]:
def get_weather(location):
    return f"The weather in {location} is sunny with low temperatures.\n"

# Inject the real observation
messages.append({
    "role": "assistant",
    "content": output.choices[0].message.content + "Observation:\n" + get_weather("London")
})

In [ ]:
final_output = client.chat.completions.create(
    messages=messages,
    max_tokens=200,
    stream=False,
    extra_body={'thinking': {'type': 'disabled'}},
)

print("Final Answer:\n", final_output.choices[0].message.content)